# Telco Customer Churn - Training Notebook

This notebook demonstrates the complete machine learning pipeline for training a customer churn prediction model using the Telco Customer Churn dataset.

The pipeline includes:
1. Data loading and preprocessing
2. Feature engineering and transformation
3. Model training (Logistic Regression)
4. Model evaluation
5. Model and preprocessor export


## 1. Setup and Imports

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix

## 2. Data Loading

In [7]:
def load_data(filepath: str) -> pd.DataFrame:
    """
    Load data from a CSV file.

    Args:
        filepath (str): Path to the CSV file.

    Returns:
        pd.DataFrame: DataFrame containing the data.
    """
    data = pd.read_csv(filepath)
    return data

# Load the dataset
data = load_data("/workspace/data/Telco-Customer-Churn.csv")

# Display basic information about the dataset
print("Dataset shape:", data.shape)
print("First 5 rows:")
display(data.head())

Dataset shape: (7043, 21)
First 5 rows:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Data Preprocessing

In [9]:
def preprocess_data(data: pd.DataFrame) -> tuple:
    """
    Preprocess data by applying encoding, scaling, and handling missing values.

    Args:
        data (pd.DataFrame): DataFrame containing raw data.

    Returns:
        tuple: (X, y, preprocessor) where X is the DataFrame of preprocessed features, 
               y is the target series, and preprocessor is the preprocessing object.
    """
    # Separate features and target
    X = data.drop(columns=["customerID", "Churn"])
    y = data["Churn"].map({"Yes": 1, "No": 0})  # Convert target to binary (1 for churn, 0 for no churn)

    # Identify categorical and numerical columns
    categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
    numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

    print(f"Categorical columns: {categorical_cols}")
    print(f"Numerical columns: {numerical_cols}")

    # Create a transformer for categorical columns
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),  # Replace missing values with the mode
            ("onehot", OneHotEncoder(handle_unknown="ignore"))  # One-hot encoding
        ]
    )

    # Create a transformer for numerical columns
    numerical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),  # Replace missing values with the median
            ("scaler", StandardScaler())  # Standardization
        ]
    )

    # Combine transformers
    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", categorical_transformer, categorical_cols),
            ("num", numerical_transformer, numerical_cols)
        ]
    )

    # Apply preprocessing
    X_processed = preprocessor.fit_transform(X)

    return X_processed, y, preprocessor

# Preprocess the data
X, y, preprocessor = preprocess_data(data)

print(f"Preprocessed data shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges']
Numerical columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges']
Preprocessed data shape: (7043, 6575)
Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64


/tmp/ipykernel_18220/3136594441.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()


## 4. Data Splitting

In [10]:
def split_data(X, y, test_size: float = 0.15, val_size: float = 0.15) -> tuple:
    """
    Split data into train, validation, and test sets.
    
    Args:
        X: Preprocessed features.
        y: Target.
        test_size (float): Size of the test set (default 0.15).
        val_size (float): Size of the validation set (default 0.15).
    
    Returns:
        tuple: (X_train, X_val, X_test, y_train, y_val, y_test)
    """
    # Split into train and test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )

    # Split train into train and validation
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=val_size, random_state=42
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

# Split the data
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

print(f"Training set size: {X_train.shape}")
print(f"Validation set size: {X_val.shape}")
print(f"Test set size: {X_test.shape}")

Training set size: (5088, 6575)
Validation set size: (898, 6575)
Test set size: (1057, 6575)


## 5. Model Training

In [11]:
def train_model(X_train, y_train) -> LogisticRegression:
    """
    Train a logistic regression model.
    
    Args:
        X_train: Training features.
        y_train: Training target.
    
    Returns:
        LogisticRegression: Trained model.
    """
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return model

# Train the model
model = train_model(X_train, y_train)
print("Model trained successfully!\n")
print(f"Model coefficients shape: {model.coef_.shape}")
print(f"Model intercept: {model.intercept_}")

Model trained successfully!

Model coefficients shape: (1, 6575)
Model intercept: [-0.83805865]


## 6. Model Evaluation

In [12]:
def evaluate_model(model, X_val, y_val) -> dict:
    """
    Evaluate the model's performance on the validation set.
    
    Args:
        model: Trained model.
        X_val: Validation features.
        y_val: Validation target.
    
    Returns:
        dict: Dictionary containing performance metrics.
    """
    # Predict probabilities and classes
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    y_pred = model.predict(X_val)

    # Calculate metrics
    metrics = {
        "roc_auc": roc_auc_score(y_val, y_pred_proba),
        "precision": precision_score(y_val, y_pred),
        "recall": recall_score(y_val, y_pred),
        "f1": f1_score(y_val, y_pred),
        "confusion_matrix": confusion_matrix(y_val, y_pred)
    }

    return metrics

# Evaluate the model on the validation set
val_metrics = evaluate_model(model, X_val, y_val)

# Evaluate the model on the test set
test_metrics = evaluate_model(model, X_test, y_test)

# Display metrics for the validation set
print("Performance metrics on the validation set:")
print(f"ROC-AUC : {val_metrics['roc_auc']:.4f}")
print(f"Precision : {val_metrics['precision']:.4f}")
print(f"Recall : {val_metrics['recall']:.4f}")
print(f"F1-score : {val_metrics['f1']:.4f}")
print(f"Confusion matrix :\n{val_metrics['confusion_matrix']}")

# Display metrics for the test set
print("\nPerformance metrics on the test set:")
print(f"ROC-AUC : {test_metrics['roc_auc']:.4f}")
print(f"Precision : {test_metrics['precision']:.4f}")
print(f"Recall : {test_metrics['recall']:.4f}")
print(f"F1-score : {test_metrics['f1']:.4f}")
print(f"Confusion matrix :\n{test_metrics['confusion_matrix']}")

Performance metrics on the validation set:
ROC-AUC : 0.8285
Precision : 0.6281
Recall : 0.5482
F1-score : 0.5855
Confusion matrix :
[[596  74]
 [103 125]]

Performance metrics on the test set:
ROC-AUC : 0.8636
Precision : 0.6820
Recall : 0.6312
F1-score : 0.6556
Confusion matrix :
[[692  83]
 [104 178]]


## 7. Model and Preprocessor Export

In [14]:
def save_artefacts(model, preprocessor, model_path: str, preprocessor_path: str) -> None:
    """
    Save the model and preprocessing to files.

    Args:
        model: Trained model.
        preprocessor: Preprocessing object.
        model_path (str): Path to save the model.
        preprocessor_path (str): Path to save the preprocessing.
    """
    joblib.dump(model, model_path)
    joblib.dump(preprocessor, preprocessor_path)
    print(f"Model saved to {model_path}")
    print(f"Preprocessing saved to {preprocessor_path}")

# Save the trained model and preprocessor
save_artefacts(
    model,
    preprocessor,
    "/workspace/models/logistic_regression_model.joblib",
    "/workspace/models/preprocessor.joblib"
)

Model saved to /workspace/models/logistic_regression_model.joblib
Preprocessing saved to /workspace/models/preprocessor.joblib


## 8. Summary and Next Steps

In this notebook, we have successfully:

1. **Loaded and explored** the Telco Customer Churn dataset
2. **Preprocessed** the data including:
   - Handling missing values with appropriate imputation strategies
   - One-hot encoding for categorical variables
   - Standard scaling for numerical variables
3. **Split** the data into training, validation, and test sets
4. **Trained** a Logistic Regression model
5. **Evaluated** the model performance using multiple metrics
6. **Exported** the trained model and preprocessor for deployment

### Next Steps:

- **Model Optimization**: Experiment with hyperparameter tuning and different algorithms
- **Feature Engineering**: Create new features that might improve model performance
- **Deployment**: Integrate the trained model into a production API
- **Monitoring**: Set up monitoring for model drift and performance degradation
- **Business Integration**: Connect the model to business processes for churn prediction and prevention